# Phase 3 — grounding & the converging-evidence decision

---

This notebook is the **heart of zlabel**. One `Labeler.label()` call runs the full six-step
annotation loop from `docs/design.md`. Phase 3 built the grounding lookups and the decision; the
separate Phase 4a later slotted the support-weighted anchor-rooted namer into steps 3–4 (the `Labeler`
here already uses them):

1. **Normalize** symbols *(Phase 2)*
2. **Score** against panels — a coarse prior *(Phase 2)*
3. **Descend** from the panel's ZFA anchor — the namer *(Phase 4a)*
4. **Guardrail (intrinsic)** — the name descends from the anchor *(Phase 4a)*
5. **Decide** — assign with confidence, roll up to a coarser tier, or abstain honestly *(Phase 3)*
6. **Emit** a `Label` evidence packet *(Phase 3)*

Phase 2 produced a ranked bucket table. Phase 3 turns that table into a **decision**: one
`Labeler(stage_hpf=…).label([...])` call runs the whole loop and returns a `Label` — or an honest
`mixed/unresolved` when the evidence does not converge.

This notebook does two things. First it **demonstrates** the loop on one keystone cluster, unfolding
every step (normalize → score → descend → decide) and every internal — `decide()`, `resolve_label()`,
and the confidence rubric. Then it hands you an **exploration tool**, `label_and_explain(markers)`,
so you can run the whole instrument on any marker list of your own.

> Section map: **1** grounding primitives · **2** the keystone, unfolded step by step · **3** the
> anchor-rooted descent (`resolve_label`) · **4** the decision ladder (`decide`) · **5** the confidence
> rubric · **6** the four honest-abstention outcomes + the forcing evidence on an abstention · **7** identity vs. state · **8** the
> `label_and_explain` exploration tool · synthesis + what's next.

Need the scoring walkthrough first? See [Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
</div>

In [1]:
import os
from pathlib import Path

import rich
from rich import inspect
from rich import pretty; pretty.install()
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.tree import Tree

import zlabel
import zlabel.label    # internal/advanced surface: decide(), normalize_markers, score_markers
import zlabel.resolve  # internal/advanced surface: resolve_label, the support-weighted anchor-rooted namer
from zlabel.ground import expression_lookup, grounds_under, stage_plausibility

# One Console drives every rich render in this notebook.
# Color key (used consistently below):
#   green  = high / agree / resolved / converged
#   yellow = medium / rollup / state
#   red    = low / mixed / abstain
#   dim    = zero / not-applicable
console = Console()
print(f"zlabel {zlabel.__version__}")

# Resolve data/ontologies relative to the repo root (the server starts there via `make notebook`).
PACKAGE = "zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / "data/ontologies"
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )

# Labeler bundles the three data authorities + panels.yaml behind one constructor. We also
# load ZFA + ZFIN expression standalone so section 1 can show the grounding primitives directly.
zfa_ontology = zlabel.load_zfa(DATA_DIR / "zfa.obo")
expression_map = zlabel.load_zfin_expression(DATA_DIR / "zfin_wildtype_expression.txt")
panels = zlabel.load_panels(Path(zlabel.__file__).parent / "panels.yaml")
muscle_anchor = next(p for p in panels if p.bucket == "muscle").ontology_anchor

lab = zlabel.Labeler(stage_hpf=48)  # 48 hpf = Long-pec, the keystone sample stage
print(f"ZFA terms: {zfa_ontology.number_of_nodes():,}  ·  genes with expression: {len(expression_map):,}")
print(f"Labeler ready (stage_hpf=48).  muscle anchor = {set(muscle_anchor)}")

zlabel 0.1.0


ZFA terms: 3,161  ·  genes with expression: 14,485
Labeler ready (stage_hpf=48).  muscle anchor = {'ZFA:0000548'}


## 1. Grounding — does a marker express where the bucket says it should?

A panel score says *which* bucket a cluster's markers belong to. **Grounding** checks that claim
against reality: do those markers actually express in the right anatomy in a live zebrafish? It is
the in-vivo evidence that lets the decision earn `high` (or refuse to).

Two pure, I/O-free functions in `ground.py` do the work:

- **`expression_lookup(expr, symbol)`** — fetch a gene's curated ZFIN wildtype-expression records
  (each one a ZFA anatomy term + a developmental-stage range).
- **`grounds_under(zfa, zfa_id, anchor)`** — walk the ZFA `is_a` / `part_of` graph to test whether
  an expression site sits *at or under* the bucket's anchor term.

(A third, `stage_plausibility`, asks whether those records fall near the sample's developmental age;
it feeds the `stage` confidence component in section 5.)

Grounding is **anchor-relative**: the same marker grounds under its own tissue and not under others —
that asymmetry is exactly what catches a panel that scored the wrong bucket.

**What to look for:** the two signatures below (kept as live `rich.inspect` peeks), then the shape of
a single `ZfinExpressionRecord`, then the punchline — `myod1` grounds under muscle but **not** under
blood.

In [2]:
rich.inspect(grounds_under, help=True, docs=True, methods=True, private=True)

╭──────────────────────────────── <function grounds_under at 0x7fbd0e3cd620> ────────────────────────────────╮
│ def grounds_under(zfa_ontology: 'nx.MultiDiGraph', rec_zfa_id: 'str', anchor: 'frozenset[str]') -> 'bool': │
│                                                                                                            │
│ Whether a ZFA anatomy term sits at or under a set of anchor terms.                                         │
│                                                                                                            │
│ Checks three cases in order: the term is in the anchor (self-match);                                       │
│ the term is not in the loaded ontology (absent -> False, not a crash);                                     │
│ the term's is_a+part_of ancestors include any anchor member.                                               │
│                                                                                                            │
│ This is a per-record predicate. label.py counts how many of the winner's                                   │
│ matched markers ground under the anchor (aggregating across records).                                      │
│                                                                                                            │
│ Args:                                                                                                      │
│     zfa_ontology (nx.MultiDiGraph): ZFA ontology graph from data.load_zfa.                                 │
│     rec_zfa_id (str): The ZFA id from a ZFIN expression record.                                            │
│     anchor (frozenset[str]): The bucket's ontology anchor ids. May be empty                                │
│         for state panels; grounds_under always returns False for an empty                                  │
│         anchor.                                                                                            │
│                                                                                                            │
│ Returns:                                                                                                   │
│     bool: True when rec_zfa_id is in anchor or is a descendant of any                                      │
│     anchor member under is_a+part_of edges.                                                                │
│                                                                                                            │
│ 38 attribute(s) not shown. Run inspect(inspect) for options.                                               │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
rich.inspect(expression_lookup, help=True, docs=True, methods=True, private=True)

╭──────────────────────────────── <function expression_lookup at 0x7fbd0e3cd580> ─────────────────────────────────╮
│ def expression_lookup(expression_map: 'Mapping[str, list[ZfinExpressionRecord]]', symbol: 'str') ->             │
│ 'list[ZfinExpressionRecord]':                                                                                   │
│                                                                                                                 │
│ Return all ZFIN wildtype-expression records for a gene symbol.                                                  │
│                                                                                                                 │
│ Lowercases the symbol before lookup to match the expression_map key convention                                  │
│ (data.load_zfin_expression lowercases keys at load time). Returns an empty                                      │
│ list when the gene has no curated expression data — not an error.                                               │
│                                                                                                                 │
│ Args:                                                                                                           │
│     expression_map (Mapping[str, list[ZfinExpressionRecord]]): From                                             │
│         data.load_zfin_expression; maps lowercased gene symbol to records.                                      │
│     symbol (str): The resolved current ZFIN symbol to look up.                                                  │
│                                                                                                                 │
│ Returns:                                                                                                        │
│     list[ZfinExpressionRecord]: All expression records for the gene, or                                         │
│     an empty list when the gene is absent from the map.                                                         │
│                                                                                                                 │
│ 38 attribute(s) not shown. Run inspect(inspect) for options.                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# The shape of one ZFA node (raw networkx attrs): name, is_a/part_of edges, xrefs.
# This is ZFA:0009258 — the anatomy term kdrl's first expression record points at (below).
zfa_ontology.nodes.get("ZFA:0009258")


{
    'name': 'angioblastic mesenchymal cell',
    'namespace': 'zebrafish_anatomy',
    'subset': ['cell_slim'],
    'synonym': ['"angioblast" EXACT []'],
    'xref': ['CL:0000566', 'TAO:0009258'],
    'is_a': ['ZFA:0009020'],
    'relationship': ['develops_from ZFA:0009081', 'end ZFS:0000044', 'start ZFS:0000000']
}

In [5]:
# The shape of one ZfinExpressionRecord: a ZFA anatomy term + a developmental-stage range.
# expression_lookup returns a list of these per gene; here, kdrl's first.
expression_map["kdrl"][0]


ZfinExpressionRecord(
    zfa_id='ZFA:0009258',
    zfa_name='angioblastic mesenchymal cell',
    start_stage='Segmentation:10-13 somites',
    end_stage='Segmentation:10-13 somites'
)

In [6]:
# myod1 is a muscle master regulator. Its in-vivo expression records land in muscle anatomy.
# Step 1 — fetch the curated ZFIN records for the resolved symbol.
records = expression_lookup(expression_map, "myod1")

# Show the first handful of distinct anatomy terms as a rich Table.
table = Table(title=f"myod1 — {len(records):,} ZFIN expression records (first distinct anatomy terms)")
table.add_column("ZFA id", style="cyan", no_wrap=True)
table.add_column("anatomy term")
seen: list[str] = []
for record in records:
    if record.zfa_id not in seen:
        seen.append(record.zfa_id)
        table.add_row(record.zfa_id, zlabel.term_name(zfa_ontology, record.zfa_id))
    if len(seen) == 8:
        break
console.print(table)

# Step 2 — grounding is anchor-relative. The SAME records ground under muscle, not under blood.
blood_anchor = next(p for p in panels if p.bucket == "blood_erythroid").ontology_anchor
under_muscle = any(grounds_under(zfa_ontology, record.zfa_id, muscle_anchor) for record in records)
under_blood = any(grounds_under(zfa_ontology, record.zfa_id, blood_anchor) for record in records)

# green = grounds (claim holds), red = does not (claim would be caught as wrong).
verdict = Table(title="grounds_under — anchor-relative", show_header=True)
verdict.add_column("anchor", no_wrap=True)
verdict.add_column("ZFA ids")
verdict.add_column("myod1 grounds?", justify="center")
verdict.add_row("muscle", str(set(muscle_anchor)), "[green]True[/green]" if under_muscle else "[red]False[/red]")
verdict.add_row("blood_erythroid", str(set(blood_anchor)), "[green]True[/green]" if under_blood else "[red]False[/red]")
console.print(verdict)
console.print(
    "[dim]myod1's records sit under the musculature-system anchor (self or ancestor) but never under "
    "the blood anchor — so a muscle panel that scored these markers is corroborated, a blood panel "
    "would not be.[/dim]"
)

myod1 — 960 ZFIN expression records
  (first distinct anatomy terms)   
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ ZFA id      ┃ anatomy term      ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ZFA:0000003 │ adaxial cell      │
│ ZFA:0001058 │ caudal fin        │
│ ZFA:0009117 │ fast muscle cell  │
│ ZFA:0001114 │ head              │
│ ZFA:0000041 │ mesoderm          │
│ ZFA:0000135 │ notochord         │
│ ZFA:0000255 │ paraxial mesoderm │
│ ZFA:0001161 │ pectoral fin      │
└─────────────┴───────────────────┘

                   grounds_under — anchor-relative                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ anchor          ┃ ZFA ids                        ┃ myod1 grounds? ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ muscle          │ {'ZFA:0000548'}                │      True      │
│ blood_erythroid │ {'ZFA:0000007', 'ZFA:0005023'} │     False      │
└─────────────────┴────────────────────────────────┴────────────────┘

myod1's records sit under the musculature-system anchor (self or ancestor) but never under the blood anchor — so a 
muscle panel that scored these markers is corroborated, a blood panel would not be.

## 2. The keystone, unfolded — the whole loop, one step at a time

`Labeler.label()` is a thin orchestration: it normalizes the markers, scores them against the
panels, runs the anchor-rooted descent, and hands everything to `decide()`. The whole call body is four
lines (see the signature peek below). To make the loop legible we run those four lines **by hand**
on one cluster and inspect each intermediate.

The **keystone** is a 48 hpf fast-muscle cluster. We define it once here and reuse it for the rest of
the notebook:

```python
KEYSTONE = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog", "kdrl"]
```

Two things make it a good worked example. The first marker uses the **old** symbol `mylz2`, which
must normalize to `mylpfa` before it can match anything — proof the normalize step earns its keep.
And the last marker, `kdrl`, is an **endothelial** gene riding along in a muscle cluster — a small
distractor the decision must not let derail the call.

The loop, top to bottom:

1. **normalize** → `mylz2` becomes `mylpfa`; resolved symbols feed both the scorer and the vote
2. **score** against panels → a ranked bucket table (the *coarse prior*, not the namer)
3. **descend** (`resolve_label`, section 3) → the deepest ZFA term the markers converge on under the panel's anchor
4. **decide** (`decide`, section 4) → assemble the `Label`, graded by the confidence rubric

**What to look for:** below, the bound-method signature; then the normalize table (watch
`mylz2 → mylpfa`).

In [7]:
rich.inspect(lab.label, help=True, docs=True, methods=True, private=True)

╭─ <bound method Labeler.label of <zlabel.label.Labeler object at 0x7fbd0d699e80>> ─╮
│ def Labeler.label(markers: 'list[str]') -> 'Label':                               │
│                                                                                   │
│ Label one cluster from its marker gene list.                                      │
│                                                                                   │
│ Normalises the markers via the loaded synonym map, scores them against            │
│ all panels, then calls decide() with the loaded ZFA ontology and ZFIN             │
│ expression data.                                                                  │
│                                                                                   │
│ Args:                                                                             │
│     markers (list[str]): Marker gene symbols ordered by significance              │
│         (rank 1 = most significant = index 0). May use old ZFIN names;            │
│         they are normalised via the GAF synonym map before scoring.               │
│                                                                                   │
│ Returns:                                                                          │
│     Label: Evidence packet with bucket, confidence, grounding evidence,           │
│     and next_step.                                                                │
│                                                                                   │
│ 28 attribute(s) not shown. Run inspect(inspect) for options.                      │
╰───────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# === Step 1 — normalize ===  (defined ONCE here; reused everywhere below)
KEYSTONE = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog", "kdrl"]  # a 48 hpf fast-muscle cluster

normalized_markers = zlabel.label.normalize_markers(KEYSTONE, lab._synonyms)

# The resolved current symbols feed both the panel scorer and the convergence descent.
# (We keep only STATUS_RESOLVED, taking the single canonical symbol for each.)
symbols = [
    next(iter(nm.symbols))
    for nm in normalized_markers
    if nm.status == zlabel.STATUS_RESOLVED
]

# Render the normalize step: input symbol -> resolved symbol(s) + status.
# green = resolved, red = unresolved.  Watch mylz2 -> mylpfa (an old name rescued).
norm_table = Table(title="Step 1 — normalize markers (input -> official ZFIN symbol)")
norm_table.add_column("rank", justify="right", style="dim")
norm_table.add_column("input")
norm_table.add_column("resolved")
norm_table.add_column("status", justify="center")
for i, nm in enumerate(normalized_markers, start=1):
    resolved = ", ".join(sorted(nm.symbols)) if nm.symbols else "—"
    ok = nm.status == zlabel.STATUS_RESOLVED
    changed = ok and resolved != nm.input
    norm_table.add_row(
        str(i),
        nm.input,
        f"[green]{resolved}[/green]" + ("  [yellow](renamed)[/yellow]" if changed else ""),
        f"[green]{nm.status}[/green]" if ok else f"[red]{nm.status}[/red]",
    )
console.print(norm_table)
console.print(f"[dim]resolved symbols passed downstream: {symbols}[/dim]")

 Step 1 — normalize markers (input -> official  
                  ZFIN symbol)                  
┏━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ rank ┃ input  ┃ resolved          ┃  status  ┃
┡━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│    1 │ mylz2  │ mylpfa  (renamed) │ resolved │
│    2 │ acta1b │ acta1b            │ resolved │
│    3 │ tnnt3a │ tnnt3a            │ resolved │
│    4 │ myod1  │ myod1             │ resolved │
│    5 │ myog   │ myog              │ resolved │
│    6 │ kdrl   │ kdrl              │ resolved │
└──────┴────────┴───────────────────┴──────────┘

resolved symbols passed downstream: ['mylpfa', 'acta1b', 'tnnt3a', 'myod1', 'myog', 'kdrl']

**What to look for (Step 2):** muscle leads by a wide margin; the single `kdrl` gives endothelium a
sliver and everything else is zero. The scorer is a *prior* — it ranks buckets but does not name the
cell. Naming is the anchor-rooted descent's job (section 3).

In [9]:
# === Step 2 — score against panels ===  (the coarse prior, NOT the namer)
scores = zlabel.label.score_markers(normalized_markers, lab._panels)

# Render the ranked bucket table. Identity buckets that matched >=1 marker, by score.
# green = the leading bucket, dim = zero-score buckets (most of the 33).
ranked = sorted(scores, key=lambda bs: -bs.score)
score_table = Table(title="Step 2 — panel scores (ranked; the coarse prior)")
score_table.add_column("bucket")
score_table.add_column("kind", style="dim")
score_table.add_column("germ layer")
score_table.add_column("score", justify="right")
score_table.add_column("matched markers")
for i, bs in enumerate(ranked):
    matched = ", ".join(mm.symbol for mm in bs.matched_markers) or "—"
    if i == 0 and bs.score > 0:
        style = "green"
    elif bs.score == 0:
        style = "dim"
    else:
        style = ""
    score_table.add_row(bs.bucket, bs.kind, bs.germ_layer or "—", f"{bs.score:.3f}", matched, style=style)
console.print(score_table)
console.print(
    "[dim]muscle dominates; the lone kdrl gives endothelium a sliver. This ranking is only the "
    "prior — the bucket NAME comes from the anchor-rooted descent next.[/dim]"
)

                     Step 2 — panel scores (ranked; the coarse prior)                      
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ bucket          ┃ kind     ┃ germ layer   ┃ score ┃ matched markers                     ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ muscle          │ identity │ mesoderm     │ 0.892 │ mylpfa, acta1b, tnnt3a, myod1, myog │
│ endothelium     │ identity │ mesoderm     │ 0.108 │ kdrl                                │
│ blood_erythroid │ identity │ mesoderm     │ 0.000 │ —                                   │
│ blood_lymphoid  │ identity │ mesoderm     │ 0.000 │ —                                   │
│ cardiac         │ identity │ mesoderm     │ 0.000 │ —                                   │
│ cartilage       │ identity │ neural crest │ 0.000 │ —                                   │
│ cycling         │ state    │ —            │ 0.000 │ —                                   │
│ endoderm_gut    │ identity │ endoderm     │ 0.000 │ —                                   │
│ epidermis       │ identity │ ectoderm     │ 0.000 │ —                                   │
│ eye             │ identity │ ectoderm     │ 0.000 │ —                                   │
│ fin             │ identity │ mesoderm     │ 0.000 │ —                                   │
│ germline        │ identity │ germline     │ 0.000 │ —                                   │
│ glia            │ identity │ ectoderm     │ 0.000 │ —                                   │
│ immune_myeloid  │ identity │ mesoderm     │ 0.000 │ —                                   │
│ interrenal      │ identity │ mesoderm     │ 0.000 │ —                                   │
│ intestine       │ identity │ endoderm     │ 0.000 │ —                                   │
│ ionocyte        │ identity │ ectoderm     │ 0.000 │ —                                   │
│ lateral_line    │ identity │ ectoderm     │ 0.000 │ —                                   │
│ liver           │ identity │ endoderm     │ 0.000 │ —                                   │
│ mesenchyme      │ identity │ mesoderm     │ 0.000 │ —                                   │
│ mural           │ identity │ mesoderm     │ 0.000 │ —                                   │
│ neural          │ identity │ ectoderm     │ 0.000 │ —                                   │
│ neural_crest    │ identity │ ectoderm     │ 0.000 │ —                                   │
│ notochord       │ identity │ mesoderm     │ 0.000 │ —                                   │
│ olfactory       │ identity │ ectoderm     │ 0.000 │ —                                   │
│ osteoblast      │ identity │ mesoderm     │ 0.000 │ —                                   │
│ otic            │ identity │ ectoderm     │ 0.000 │ —                                   │
│ pancreas        │ identity │ endoderm     │ 0.000 │ —                                   │
│ pigment         │ identity │ neural crest │ 0.000 │ —                                   │
│ pineal          │ identity │ ectoderm     │ 0.000 │ —                                   │
│ pituitary       │ identity │ ectoderm     │ 0.000 │ —                                   │
│ pronephros      │ identity │ mesoderm     │ 0.000 │ —                                   │
│ stress_response │ state    │ —            │ 0.000 │ —                                   │
└─────────────────┴──────────┴──────────────┴───────┴─────────────────────────────────────┘

muscle dominates; the lone kdrl gives endothelium a sliver. This ranking is only the prior — the bucket NAME comes 
from the anchor-rooted descent next.

## 3. The anchor-rooted descent — `resolve_label` names the cell *(Phase 4a)*

The panels rank buckets; they do not say *how specific* the call can be. That is the namer's job.
`resolve.resolve_label(symbols, …, anchor=…)` finds the most specific ZFA anatomy term the cluster's
markers **actually converge on in vivo**, by descending from the winning panel's ontology anchor:

1. **Tally with ancestor credit.** Each distinct resolved gene looks up its ZFIN expression records.
   For every anatomy term a gene expresses in, it credits that term **and all of its `is_a` /
   `part_of` ancestors** — a gene in *posterior hypaxial muscle* also credits *muscle*, *trunk
   musculature*, *musculature system*, and so on up the DAG. One vote per gene per term (a gene with
   many records can't stuff the ballot).
2. **Seed at the panel's anchor.** The descent starts at the winning bucket's `ontology_anchor` (here
   *musculature system*), provided at least `CONVERGENCE_MIN` (= 3) genes support it.
3. **Roll down the best-supported child.** At each step it moves into the child term the most genes
   support — scored support × IC, with support the dominant signal and IC only tilting ties toward
   specificity. It keeps descending while the child still keeps `≥ CONVERGENCE_MIN` genes, retains
   `≥ DESCENT_SUPPORT_FRACTION` (= 0.6) of its parent's support, and **uniquely** leads its siblings.
   A support tie means the markers have split across subtypes, so the walk stops.
4. **Stop at the deepest agreed term.** That terminal term names the cell; its `genes` are the
   **convergent genes** on the `Label`. `STOPLIST` roots (*whole organism* …) are never seeded or
   entered, and an unsupported anchor returns nothing — the decision then falls back to the coarse
   panel bucket.

Because the name descends *from* the anchor, it is at or under it by construction: the old post-hoc
guardrail is folded into the walk, so a contradiction is impossible. IC is no longer a gate — the
relative-support floor in step 3 is what stops a coincidental few-gene leaf.

**What to look for:** the terminal term the descent returns for the keystone below — **muscle** (the
tissue, `ZFA:0005145`), reached by descending from the *musculature system* anchor. The markers don't
converge on a single finer muscle subtype, so the walk holds at the broad muscle tissue rather than
over-specifying. Because the descent is anchored at the muscle anchor, the endothelial rider `kdrl`
can't pull the call into vasculature — its vascular-specific terms lie off the anchor's path; it only
credits the broad *muscle* term that muscle tissue's own vasculature shares.

In [10]:
# === Step 3 — the anchor-rooted descent ===  (resolve.resolve_label, seeded at the muscle anchor)
# The descent needs the winning panel's anchor as its root; for the keystone that is the muscle anchor.
votes = zlabel.resolve.resolve_label(
    symbols,
    expression_map=lab._expression_map,
    zfa_ontology=lab._zfa_ontology,
    information_content=lab._information_content,
    anchor=muscle_anchor,
)

# resolve_label returns a one-element list: the terminal term the descent stopped at (or empty when
# the anchor itself is unsupported -> the decision falls back to the coarse panel bucket).
if votes:
    winner = votes[0]
    vote_table = Table(title="Step 3 — the descent's terminal term (named from the muscle anchor)")
    vote_table.add_column("ZFA term")
    vote_table.add_column("IC (bits)", justify="right")
    vote_table.add_column("genes", justify="right")
    vote_table.add_column("convergent genes")
    vote_table.add_row(
        f"{winner.zfa_name}  [dim]{winner.zfa_id}[/dim]",
        f"{winner.information_content:.2f}",
        str(len(winner.genes)),
        ", ".join(winner.genes),
    )
    console.print(vote_table)
    console.print(
        f"[green]named -> {winner.zfa_name}[/green]  "
        f"[dim](descended from the musculature-system anchor; {len(winner.genes)} convergent genes. "
        f"The markers don't converge on a finer subtype, so the walk holds at the broad muscle tissue.)[/dim]"
    )
else:
    console.print("[red]the descent found no supported term under the muscle anchor -> panel fallback[/red]")

          Step 3 — the descent's terminal term (named from the muscle anchor)          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ZFA term            ┃ IC (bits) ┃ genes ┃ convergent genes                          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ muscle  ZFA:0005145 │      2.75 │     6 │ acta1b, kdrl, mylpfa, myod1, myog, tnnt3a │
└─────────────────────┴───────────┴───────┴───────────────────────────────────────────┘

named -> muscle  (descended from the musculature-system anchor; 6 convergent genes. The markers don't converge on a
finer subtype, so the walk holds at the broad muscle tissue.)

## 4. The decision — `decide()` assembles the `Label` *(Phase 3)*

`decide()` is the pure, I/O-free core. It takes the score table plus the already-loaded grounding
data and returns a `Label`. It runs three stages in order:

**Stage A/B — prechecks (abstain early).** Before doing any work it refuses the hopeless cases.
*(A)* if no marker hit any **identity** panel at all, abstain (`provisional`). *(B)* if the top
identity bucket's adjusted score is below `MIN_SIGNAL` (= 0.15) — the markers barely touch any
bucket — abstain (`provisional`). No least-bad guessing.

**Stage C/D — the dominance ladder.** Compare the top two identity buckets by adjusted score.
- *(C) Dominant winner* — gap `≥ DOMINANCE_GAP` (= 0.30), or only one identity bucket matched: this
  is a confident single call. `decide()` runs the anchor-rooted descent (section 3) to **name** it.
  Because the descent starts *at* the winning panel's ontology anchor, the name sits under it by
  construction — the guardrail is intrinsic, with no separate discard step; if the markers do not
  support the anchor at all, the descent returns nothing and the call falls back to the coarse panel
  bucket.
- *(D) Near-tie* — the runner-up is within `DOMINANCE_GAP`. If all the near-tied contenders share
  one germ layer, **roll up** to that germ layer (`underclustered`) rather than guess between them.
  If they span **contradictory** germ layers (e.g. neural vs. blood), abstain (`mixed/unresolved`).

**The confidence rubric (section 5).** An assigned call is then graded on four components —
coherence, margin, grounding, stage — combined into a 0–1 score and read off a fixed tier ladder.
Two caps keep it honest: the **convergence cap** (a single-bucket `high` needs real grounding/stage
corroboration; strong panels alone top out at medium) and the **rollup cap** (a germ-layer rollup
never exceeds medium). The floor: any assigned label is at least `low`.

For the keystone this is the easy path: muscle dominates (Stage C), the descent names *muscle* (the
tissue, sitting under the muscle anchor by construction), and every rubric component is strong → a
clean `high`.

**What to look for:** the assembled `Label` as a panel below — bucket, depth, the four confidence
components, the convergent genes, and the one-line rationale — then its broad→specific levels chain
as a tree.

In [11]:
# === Step 4 — decide ===  (the pure core: score table + grounding data -> Label)
# We call decide() directly with the Labeler's loaded data, exactly as lab.label() does internally.
# marker_specificity is the per-gene panel-IDF: it lets a single sharply lineage-specific marker
# rescue an otherwise weak-signal cluster (the specificity rescue). lab.label() passes it too.
label = zlabel.label.decide(
    scores,
    anchors=lab._anchors,
    expression_map=lab._expression_map,
    zfa_ontology=lab._zfa_ontology,
    stage_hpf=lab.stage_hpf,
    symbols=symbols,
    information_content=lab._information_content,
    marker_specificity=lab._marker_specificity,
)

# Two small renderers, reused later by the label_and_explain() exploration tool.
TIER_COLOR = {"high": "green", "medium": "yellow", "low": "red", None: "red"}


def render_label_panel(label: zlabel.Label) -> Panel:
    """Render a Label evidence packet as a tier-colored rich.Panel.

    Args:
        label (zlabel.Label): The evidence packet from decide()/Labeler.label().

    Returns:
        Panel: A bordered summary (bucket, depth, confidence + the 4 components,
        convergent genes, rationale), colored by confidence tier. On an abstention it
        shows the forcing evidence (ood / margin / candidates) instead — see section 6.
    """
    color = TIER_COLOR.get(label.confidence, "red")
    if label.abstained:
        cands = " · ".join(f"{c.bucket} (Δ{c.margin_to_top:.2f})" for c in label.candidates) or "none"
        body = (
            f"[red]ABSTAINED[/red] — {label.rationale}\n"
            f"ambiguity flag   {label.ambiguity_flag}\n"
            f"forcing evidence ood=[b]{label.ood}[/b]   margin={label.margin:.3f}\n"
            f"candidates       [dim]{cands}[/dim]"
        )
    else:
        comps = "   ".join(f"{k}={v:.2f}" for k, v in label.confidence_components.items())
        convergent = ", ".join(label.convergent_genes) or "[dim](panel fallback — none)[/dim]"
        body = (
            f"bucket           [b]{label.bucket}[/b]   [dim]({label.zfa_id})[/dim]\n"
            f"depth            {label.depth}   ([dim]evidence-derived[/dim])\n"
            f"panel prior      {label.panel_bucket}  /  {label.panel_germ_layer}\n"
            f"confidence       [{color}]{label.confidence}[/{color}]  (score {label.confidence_score:.3f})\n"
            f"components        {comps}\n"
            f"convergent genes {convergent}\n"
            f"next step        {label.next_step}\n"
            f"rationale        [dim]{label.rationale}[/dim]"
        )
    return Panel(body, title=f"Label — {label.bucket}", border_style=color, expand=False)


def render_levels_tree(label: zlabel.Label) -> Tree:
    """Render a Label's broad->specific levels chain as a rich.Tree.

    Args:
        label (zlabel.Label): The evidence packet whose levels to display.

    Returns:
        Tree: One nested node per level, broadest at the root, leaf in green.
    """
    if not label.levels:
        return Tree("[red](no levels — abstained)[/red]")
    tree = Tree(f"[dim]{label.levels[0]}[/dim]")
    node = tree
    for level in label.levels[1:-1]:
        node = node.add(level)
    node.add(f"[green]{label.levels[-1]}[/green]")  # the named, most-specific term
    return tree


console.print(render_label_panel(label))
console.print("[b]levels chain[/b] (broad -> specific):")
console.print(render_levels_tree(label))

╭──────────────────────────────── Label — muscle ────────────────────────────────╮
│ bucket           muscle   (ZFA:0005145)                                        │
│ depth            3   (evidence-derived)                                        │
│ panel prior      muscle  /  mesoderm                                           │
│ confidence       high  (score 1.000)                                           │
│ components        coherence=1.00   margin=1.00   grounding=1.00   stage=1.00   │
│ convergent genes acta1b, kdrl, mylpfa, myod1, myog, tnnt3a                     │
│ next step        subcluster                                                    │
│ rationale        muscle (IC 2.75) -- 6/6 markers converge; panel prior: muscle │
╰────────────────────────────────────────────────────────────────────────────────╯

levels chain (broad -> specific):

musculature system
└── portion of tissue
    └── muscle

## 5. The confidence rubric — how a call earns its tier

An assigned call is graded on a weighted 0–1 score over four components, each clipped to `[0, 1]`:

| component | weight | what it measures |
|-----------|:------:|------------------|
| **coherence** | 0.40 | rank-weighted strength of the winner's matched markers (breadth saturates at ~3 top markers) |
| **margin** | 0.30 | how far the winner leads the runner-up, normalized by `DOMINANCE_GAP` |
| **grounding** | 0.20 | fraction of the winner's gradable markers whose ZFIN expression grounds under the named ZFA term (or the panel anchor on fallback) |
| **stage** | 0.10 | fraction of those markers on-stage for the sample (`NEUTRAL = 0.5` when stage isn't gradable) |

The weights sum to 1.0, so the result is always a clean `0–1` number, read off a fixed ladder:
`≥ 0.80` → **high**, `≥ 0.60` → **medium**, else **low**.

`confidence = 0.40·coherence + 0.30·margin + 0.20·grounding + 0.10·stage`

Two caps then override the raw arithmetic — this is where the honesty lives:

- **convergence cap** — a single-bucket `high` needs *real* grounding/stage corroboration
  (`_supports_high`): strong panels alone top out at **medium**, and anatomy that *contradicts* the
  call blocks `high` no matter how the sum lands (anatomy is the stronger anchor; stage is the soft
  0.10 signal).
- **rollup cap** — a germ-layer rollup never reports above **medium** (it makes no single-anatomy
  claim).

The weights are provisional; **Phase 4b measures the baseline** — calibration against held-out labels is deferred.

**What to look for:** the keystone's rubric below. Every component is strong — coherence and margin
maxed, all convergent markers ground to muscle anatomy on-stage at Long-pec — so the weighted sum is
1.00 and clears `high` cleanly, with the convergence cap satisfied.

In [12]:
# The four weights (sum to 1.0) — mirrors W_COHERENCE/W_MARGIN/W_GROUNDING/W_STAGE in label.py.
WEIGHTS = {"coherence": 0.40, "margin": 0.30, "grounding": 0.20, "stage": 0.10}


def render_rubric(label: zlabel.Label) -> Table:
    """Render a Label's confidence rubric as a component/weight/value/bar/contribution table.

    Args:
        label (zlabel.Label): An assigned (non-abstained) Label.

    Returns:
        Table: One row per component plus a footer row with the weighted score
        and its tier (tier-colored).
    """
    color = TIER_COLOR.get(label.confidence, "red")
    rubric = Table(title=f"confidence rubric -> {label.confidence} ({label.confidence_score:.3f})")
    rubric.add_column("component")
    rubric.add_column("weight", justify="right")
    rubric.add_column("value", justify="right")
    rubric.add_column("bar")
    rubric.add_column("contribution", justify="right")
    for name, w in WEIGHTS.items():
        v = label.confidence_components[name]
        bar = "█" * round(v * 16)  # 16-wide unit bar
        rubric.add_row(name, f"{w:.2f}", f"{v:.2f}", f"[cyan]{bar}[/cyan]", f"{w * v:.2f}")
    rubric.add_section()
    rubric.add_row("[b]score[/b]", "", "", "", f"[{color}]{label.confidence_score:.2f}[/{color}]")
    return rubric


console.print(render_rubric(label))
console.print(
    "[dim]Tiers: >= 0.80 high · >= 0.60 medium · else low.  Caps (honest by design): a germ-layer "
    "rollup never exceeds medium, and a high needs grounding/stage corroboration — contradicting "
    "anatomy blocks high regardless of the sum. Weights are provisional; Phase 4b measures the "
    "baseline (calibration deferred).[/dim]"
)

               confidence rubric -> high (1.000)                
┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ component ┃ weight ┃ value ┃ bar              ┃ contribution ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ coherence │   0.40 │  1.00 │ ████████████████ │         0.40 │
│ margin    │   0.30 │  1.00 │ ████████████████ │         0.30 │
│ grounding │   0.20 │  1.00 │ ████████████████ │         0.20 │
│ stage     │   0.10 │  1.00 │ ████████████████ │         0.10 │
├───────────┼────────┼───────┼──────────────────┼──────────────┤
│ score     │        │       │                  │         1.00 │
└───────────┴────────┴───────┴──────────────────┴──────────────┘

Tiers: >= 0.80 high · >= 0.60 medium · else low.  Caps (honest by design): a germ-layer rollup never exceeds 
medium, and a high needs grounding/stage corroboration — contradicting anatomy blocks high regardless of the sum. 
Weights are provisional; Phase 4b measures the baseline (calibration deferred).

## 6. It never overcalls — the four honest outcomes

The reason to ground at all is to know when *not* to commit. The same `label()` call produces four
qualitatively different outcomes depending on how the evidence falls — this is the payoff of the
dominance ladder in section 4:

- **confident** (`none`) — one bucket dominates and grounds → a named, `high` call.
- **abstain** (`mixed/unresolved`) — the near-tied contenders span **contradictory** germ layers →
  refuse to choose.
- **roll up** (`underclustered`) — no single bucket dominates, but the contenders **share** a germ
  layer → back off to that coarser, honest tier instead of guessing (capped at medium).
- **assign anyway, hedged** — a single weak-but-real marker is still labeled, just never `high`.

**What to look for:** four marker sets, one per outcome, in a tier-colored table — green confident,
red mixed/abstain, yellow rollup, and a hedged single-signal call. Watch the `flag` and `confidence`
columns track the evidence.

In [13]:
# One marker set per outcome of the dominance ladder. KEYSTONE[:5] drops the kdrl distractor
# so the "confident" row is the pure muscle case.
cases = {
    "confident (muscle)":      KEYSTONE[:5],                         # one bucket dominates -> named, high
    "abstain (mixed)":         ["elavl3", "neurod1", "gata1a", "hbae1.1"],  # neural (ectoderm) + blood (mesoderm)
    "rollup (underclustered)": ["myod1", "myog", "gata1a", "hbae1.1"],      # muscle + blood, both mesoderm
    "weak single signal":      ["elavl3"],                           # lone neural marker -> assigned, hedged
}

# Color each row by its honest outcome (matches the notebook color key):
#   green = confident/named · red = mixed/abstain · yellow = rollup · tier-color for the hedged call.
FLAG_COLOR = {"none": "green", "mixed": "red", "underclustered": "yellow", "provisional": "red"}

honest = Table(title="four honest outcomes of the same label() call")
honest.add_column("case")
honest.add_column("bucket")
honest.add_column("confidence", justify="center")
honest.add_column("flag")
honest.add_column("markers")
for name, markers in cases.items():
    r = lab.label(markers)
    # Confident single-bucket calls use the tier color; abstain/rollup use the flag color.
    style = TIER_COLOR.get(r.confidence, "red") if r.ambiguity_flag == "none" else FLAG_COLOR.get(r.ambiguity_flag, "red")
    honest.add_row(name, r.bucket, (r.confidence or "—"), r.ambiguity_flag, ", ".join(markers), style=style)
console.print(honest)
console.print(
    "[dim]neural + blood span contradictory germ layers -> honest mixed/unresolved. muscle + blood "
    "share mesoderm but neither dominates -> roll up to mesoderm (capped medium). a lone neural "
    "marker is still assigned (neural) — at medium, never high.[/dim]"
)

                                   four honest outcomes of the same label() call                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ case                    ┃ bucket             ┃ confidence ┃ flag           ┃ markers                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ confident (muscle)      │ musculature system │    high    │ none           │ mylz2, acta1b, tnnt3a, myod1, myog │
│ abstain (mixed)         │ mixed/unresolved   │     —      │ mixed          │ elavl3, neurod1, gata1a, hbae1.1   │
│ rollup (underclustered) │ mesoderm           │   medium   │ underclustered │ myod1, myog, gata1a, hbae1.1       │
│ weak single signal      │ neural             │   medium   │ none           │ elavl3                             │
└─────────────────────────┴────────────────────┴────────────┴────────────────┴────────────────────────────────────┘

neural + blood span contradictory germ layers -> honest mixed/unresolved. muscle + blood share mesoderm but neither
dominates -> roll up to mesoderm (capped medium). a lone neural marker is still assigned (neural) — at medium, 
never high.

### Forcing evidence — what the engine hands you on an abstention

An abstention is a refusal to commit, not a dead end. Even when `label()` declines, the `Label` still
carries the **forcing evidence** a caller needs to decide whether to override the engine and take the
top candidate anyway — the engine stays honest, the caller owns the risk:

- **`ood`** — is the top candidate even reachable? `in_set` (force-able, but a broad attractor can
  mask a blind spot — a *soft* signal), `structural` (the evidence converges nowhere reachable — a
  real blind spot, high-precision *don't force*), `doublet` (contradictory germ layers — a probable
  mis-cluster the curator should *split*, not relabel), `no_signal` (no identity panel hit at all).
- **`margin`** — the raw, un-clamped lead of the top adjusted score over the runner-up. Low ⇒ a true
  near-tie. (Distinct from the clamped `margin` confidence *component* in section 5.)
- **`candidates`** — the near-tie band, best-first, each with its `margin_to_top`. Size ≥ 2 ⇒ contested.

**What to look for:** the same four cases, now showing their forcing evidence. The confident muscle
call is `in_set` with the maximum margin; the mixed neural+blood case is a `doublet` — its two
candidates span contradictory germ layers, so a caller should split the cluster rather than force a
call.

In [14]:
# When the engine abstains, the Label still carries the FORCING EVIDENCE a caller needs to decide
# whether to override it and take the top candidate anyway. The engine stays honest; the caller owns
# the risk. Three fields carry it: ood (is the top candidate reachable?), margin (the raw lead), and
# candidates (the near-tie band, best-first, each with its margin to the top).
OOD_COLOR = {"in_set": "yellow", "structural": "red", "doublet": "red", "no_signal": "dim"}

forcing = Table(title="forcing evidence — the same four cases")
forcing.add_column("case")
forcing.add_column("abstained", justify="center")
forcing.add_column("ood")
forcing.add_column("margin", justify="right")
forcing.add_column("candidates (Δ to top)")
for name, markers in cases.items():
    lb = lab.label(markers)
    cands = " · ".join(f"{c.bucket} [dim]Δ{c.margin_to_top:.2f}[/dim]" for c in lb.candidates) or "[dim]—[/dim]"
    oc = OOD_COLOR.get(lb.ood, "white")
    forcing.add_row(
        name,
        "[red]yes[/red]" if lb.abstained else "[green]no[/green]",
        f"[{oc}]{lb.ood}[/{oc}]",
        f"{lb.margin:.3f}",
        cands,
    )
console.print(forcing)
console.print(
    "[dim]in_set is force-able but soft (a broad attractor can still hide a blind spot); structural "
    "and doublet are high-precision 'do not force' (a real blind spot / a probable mis-cluster); "
    "no_signal means nothing matched. The mixed neural+blood case is a doublet — split it, don't "
    "relabel.[/dim]"
)

                             forcing evidence — the same four cases                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ case                    ┃ abstained ┃ ood     ┃ margin ┃ candidates (Δ to top)                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ confident (muscle)      │    no     │ in_set  │  1.000 │ muscle Δ0.00                         │
│ abstain (mixed)         │    yes    │ doublet │  0.273 │ neural Δ0.00 · blood_erythroid Δ0.27 │
│ rollup (underclustered) │    no     │ in_set  │  0.273 │ muscle Δ0.00 · blood_erythroid Δ0.27 │
│ weak single signal      │    no     │ in_set  │  1.000 │ neural Δ0.00                         │
└─────────────────────────┴───────────┴─────────┴────────┴──────────────────────────────────────┘

in_set is force-able but soft (a broad attractor can still hide a blind spot); structural and doublet are 
high-precision 'do not force' (a real blind spot / a probable mis-cluster); no_signal means nothing matched. The 
mixed neural+blood case is a doublet — split it, don't relabel.

## 7. Identity vs. state (optional)

Cell **identity** (what a cell *is*) and cell **state** (what it is *doing*) are orthogonal. A
dividing muscle cell is still muscle, so detected state programs (cycling, stress) are reported
*alongside* the identity call on every `Label` — never in competition with it. A state panel is only
flagged when it clears both `STATE_MIN_WEIGHT` and `N_STATE_MIN` (≥ 2 distinct markers), so a single
`mki67` hit won't trip it.

**What to look for:** muscle markers plus three cell-cycle genes. The identity call stays muscle; the
cycling program shows up in `states`, in yellow.

In [15]:
# Muscle markers + cell-cycle markers (mki67, pcna, top2a hit the cycling state panel).
r = lab.label(["mylz2", "acta1b", "myod1", "mki67", "pcna", "top2a"])

states_txt = ("[yellow]" + ", ".join(r.states) + "[/yellow]") if r.states else "[dim](none)[/dim]"
console.print(
    Panel(
        f"identity bucket  [b]{r.bucket}[/b]  ([{TIER_COLOR.get(r.confidence, 'red')}]{r.confidence}[/])\n"
        f"states           {states_txt}",
        title="identity vs. state",
        border_style="green",
        expand=False,
    )
)
console.print("[dim]The cycling program is reported as a state; the identity call stays muscle.[/dim]")

╭──────────── identity vs. state ─────────────╮
│ identity bucket  musculature system  (high) │
│ states           cycling                    │
╰─────────────────────────────────────────────╯

The cycling program is reported as a state; the identity call stays muscle.

## 8. Explore it yourself — `label_and_explain(markers)`

Everything above unfolded one keystone by hand. This section packages that whole unfold into a single
reusable instrument: **`label_and_explain(markers, stage_hpf=48)`** runs the full loop on *any*
marker list and renders every step — the normalize table, the panel scores, the descent's terminal
term, the grounding evidence, the confidence rubric, and the final `Label` panel + levels tree.

It threads `lab`'s already-loaded data (ZFA, ZFIN expression, panels, IC model) straight through, so
it is fast and produces exactly what `lab.label(markers)` would decide — just with the internals laid
bare. Pass a different `stage_hpf` to see the stage component move.

**What to look for:** below, the tool re-derives the keystone (identical result to sections 2–5),
proving the helper faithfully reproduces the loop. Then the next cell turns it loose on the
abstain / rollup / mixed marker sets — run it, then edit `MY_CLUSTER` with your own genes.

In [16]:
def label_and_explain(markers: list[str], stage_hpf: float | None = 48) -> zlabel.Label:
    """Run the full label() loop on any marker list and render every step.

    Threads the module-level Labeler `lab`'s already-loaded data (ZFA, ZFIN
    expression, panels, IC model, marker-specificity) through the same four steps
    lab.label() runs internally — normalize, score, vote, decide — printing each
    intermediate with rich. The returned Label is identical to lab.label(markers)
    when stage_hpf matches lab.stage_hpf.

    Args:
        markers (list[str]): Marker gene symbols, most significant first. May use
            old ZFIN names; they are normalized before scoring.
        stage_hpf (float | None): Developmental stage in hpf for the stage
            component. Defaults to 48 (Long-pec). None disables stage grounding.

    Returns:
        zlabel.Label: The evidence packet for this cluster.
    """
    console.rule(f"[b]label_and_explain[/b]  ({len(markers)} markers, stage_hpf={stage_hpf})")

    # --- Step 1: normalize ---
    nms = zlabel.label.normalize_markers(markers, lab._synonyms)
    syms = [next(iter(nm.symbols)) for nm in nms if nm.status == zlabel.STATUS_RESOLVED]
    t = Table(title="1 · normalize")
    t.add_column("input"); t.add_column("resolved"); t.add_column("status", justify="center")
    for nm in nms:
        ok = nm.status == zlabel.STATUS_RESOLVED
        resolved = ", ".join(sorted(nm.symbols)) if nm.symbols else "—"
        t.add_row(nm.input, f"[green]{resolved}[/green]" if ok else resolved,
                  f"[green]{nm.status}[/green]" if ok else f"[red]{nm.status}[/red]")
    console.print(t)

    # --- Step 2: score against panels ---
    sc = zlabel.label.score_markers(nms, lab._panels)
    t = Table(title="2 · panel scores (non-zero buckets)")
    t.add_column("bucket"); t.add_column("kind", style="dim"); t.add_column("germ layer")
    t.add_column("score", justify="right"); t.add_column("matched")
    for bs in sorted([b for b in sc if b.score > 0], key=lambda b: -b.score):
        style = "green" if bs.kind == "identity" else "yellow"  # state panels in yellow
        t.add_row(bs.bucket, bs.kind, bs.germ_layer or "—", f"{bs.score:.3f}",
                  ", ".join(mm.symbol for mm in bs.matched_markers), style=style)
    console.print(t)

    # --- Step 3: anchor-rooted descent ---  (seed at the winning identity bucket's anchor)
    top_identity = next((b for b in sorted(sc, key=lambda b: -b.score) if b.kind == "identity" and b.score > 0), None)
    anchor = lab._anchors.get(top_identity.bucket, frozenset()) if top_identity else frozenset()
    vs = zlabel.resolve.resolve_label(syms, expression_map=lab._expression_map, zfa_ontology=lab._zfa_ontology,
                                      information_content=lab._information_content, anchor=anchor)
    if vs:
        v = vs[0]  # the terminal term the descent stopped at
        t = Table(title="3 · anchor-rooted descent (terminal term)")
        t.add_column("ZFA term"); t.add_column("IC", justify="right")
        t.add_column("genes", justify="right"); t.add_column("convergent genes")
        t.add_row(f"{v.zfa_name} [dim]{v.zfa_id}[/dim]", f"{v.information_content:.2f}",
                  str(len(v.genes)), ", ".join(v.genes))
        console.print(t)
    else:
        console.print("[red]3 · anchor-rooted descent: no supported term under the anchor -> panel fallback[/red]")

    # --- Step 4: decide -> Label ---  (marker_specificity enables the single-marker specificity rescue)
    out = zlabel.label.decide(sc, anchors=lab._anchors, expression_map=lab._expression_map,
                              zfa_ontology=lab._zfa_ontology, stage_hpf=stage_hpf,
                              symbols=syms, information_content=lab._information_content,
                              marker_specificity=lab._marker_specificity)

    # Grounding evidence (the markers whose in-vivo anatomy supports the call).
    if out.expression_evidence:
        t = Table(title="4 · grounding evidence (marker -> grounded anatomy)")
        t.add_column("symbol"); t.add_column("ZFA id", style="cyan"); t.add_column("anatomy")
        for hit in out.expression_evidence:
            t.add_row(hit.symbol, hit.zfa_id, hit.zfa_name)
        console.print(t)

    # Confidence rubric (skip when abstained — no graded components).
    if not out.abstained:
        console.print(render_rubric(out))

    # Final Label + levels chain. (On an abstention the panel shows the forcing evidence — section 6.)
    console.print(render_label_panel(out))
    if out.levels:
        console.print(render_levels_tree(out))
    return out


# Re-derive the keystone through the tool: identical decision to sections 2–5, internals laid bare.
_ = label_and_explain(KEYSTONE, stage_hpf=48)

────────────────────────────────── label_and_explain  (6 markers, stage_hpf=48) ───────────────────────────────────

         1 · normalize          
┏━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ input  ┃ resolved ┃  status  ┃
┡━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ mylz2  │ mylpfa   │ resolved │
│ acta1b │ acta1b   │ resolved │
│ tnnt3a │ tnnt3a   │ resolved │
│ myod1  │ myod1    │ resolved │
│ myog   │ myog     │ resolved │
│ kdrl   │ kdrl     │ resolved │
└────────┴──────────┴──────────┘

                         2 · panel scores (non-zero buckets)                         
┏━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ bucket      ┃ kind     ┃ germ layer ┃ score ┃ matched                             ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ muscle      │ identity │ mesoderm   │ 0.892 │ mylpfa, acta1b, tnnt3a, myod1, myog │
│ endothelium │ identity │ mesoderm   │ 0.108 │ kdrl                                │
└─────────────┴──────────┴────────────┴───────┴─────────────────────────────────────┘

                    3 · anchor-rooted descent (terminal term)                    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ZFA term           ┃   IC ┃ genes ┃ convergent genes                          ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ muscle ZFA:0005145 │ 2.75 │     6 │ acta1b, kdrl, mylpfa, myod1, myog, tnnt3a │
└────────────────────┴──────┴───────┴───────────────────────────────────────────┘

    4 · grounding evidence (marker -> grounded    
                     anatomy)                     
┏━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ symbol ┃ ZFA id      ┃ anatomy                 ┃
┡━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ mylpfa │ ZFA:0009117 │ fast muscle cell        │
│ acta1b │ ZFA:0001056 │ myotome                 │
│ tnnt3a │ ZFA:0001085 │ hypaxial myotome region │
│ myod1  │ ZFA:0009117 │ fast muscle cell        │
│ myog   │ ZFA:0009117 │ fast muscle cell        │
└────────┴─────────────┴─────────────────────────┘

               confidence rubric -> high (1.000)                
┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ component ┃ weight ┃ value ┃ bar              ┃ contribution ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ coherence │   0.40 │  1.00 │ ████████████████ │         0.40 │
│ margin    │   0.30 │  1.00 │ ████████████████ │         0.30 │
│ grounding │   0.20 │  1.00 │ ████████████████ │         0.20 │
│ stage     │   0.10 │  1.00 │ ████████████████ │         0.10 │
├───────────┼────────┼───────┼──────────────────┼──────────────┤
│ score     │        │       │                  │         1.00 │
└───────────┴────────┴───────┴──────────────────┴──────────────┘

╭──────────────────────────────── Label — muscle ────────────────────────────────╮
│ bucket           muscle   (ZFA:0005145)                                        │
│ depth            3   (evidence-derived)                                        │
│ panel prior      muscle  /  mesoderm                                           │
│ confidence       high  (score 1.000)                                           │
│ components        coherence=1.00   margin=1.00   grounding=1.00   stage=1.00   │
│ convergent genes acta1b, kdrl, mylpfa, myod1, myog, tnnt3a                     │
│ next step        subcluster                                                    │
│ rationale        muscle (IC 2.75) -- 6/6 markers converge; panel prior: muscle │
╰────────────────────────────────────────────────────────────────────────────────╯

musculature system
└── portion of tissue
    └── muscle

### Now try your own cluster

The cell below seeds `label_and_explain` with marker sets that hit each honest outcome — a confident
endothelial call, a germ-layer **rollup**, and a contradictory **mixed/unresolved** abstention.
Run it to watch the loop diverge on each, then point `MY_CLUSTER` at your own genes (old or current
ZFIN symbols both work) and re-run.

**What to look for:** the same six-panel readout for every input — only the *outcome* changes. Notice
how the anchor-rooted descent and the rubric react: a clean cluster names a deep ZFA term and earns a high
score; a mixed cluster collapses to an abstention with no rubric at all.

In [17]:
# Seeded marker sets — one per honest outcome. Edit MY_CLUSTER (last) with your own genes.
SEEDS = {
    "confident endothelium": ["kdrl", "fli1a", "cdh5", "pecam1"],          # one tight bucket -> named, high
    "rollup (mesoderm)":     ["myod1", "myog", "gata1a", "hbae1.1"],       # muscle + blood share mesoderm
    "mixed/unresolved":      ["elavl3", "neurod1", "gata1a", "hbae1.1"],   # neural + blood: contradictory
}

# Your turn: replace these symbols and re-run the cell.
MY_CLUSTER = ["kdrl", "fli1a", "cdh5", "pecam1"]

for name, markers in SEEDS.items():
    console.print(f"\n[b magenta]» {name}[/b magenta]")
    label_and_explain(markers, stage_hpf=48)

console.print("\n[b magenta]» MY_CLUSTER (edit me)[/b magenta]")
my_label = label_and_explain(MY_CLUSTER, stage_hpf=48)

» confident endothelium

────────────────────────────────── label_and_explain  (4 markers, stage_hpf=48) ───────────────────────────────────

         1 · normalize          
┏━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ input  ┃ resolved ┃  status  ┃
┡━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ kdrl   │ kdrl     │ resolved │
│ fli1a  │ fli1     │ resolved │
│ cdh5   │ cdh5     │ resolved │
│ pecam1 │ pecam1a  │ resolved │
└────────┴──────────┴──────────┘

                    2 · panel scores (non-zero buckets)                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ bucket      ┃ kind     ┃ germ layer ┃ score ┃ matched                   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ endothelium │ identity │ mesoderm   │ 1.000 │ kdrl, fli1, cdh5, pecam1a │
└─────────────┴──────────┴────────────┴───────┴───────────────────────────┘

                   3 · anchor-rooted descent (terminal term)                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ ZFA term                                  ┃   IC ┃ genes ┃ convergent genes ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ blood vessel endothelial cell ZFA:0009036 │ 8.15 │     3 │ cdh5, fli1, kdrl │
└───────────────────────────────────────────┴──────┴───────┴──────────────────┘

...

                   3 · anchor-rooted descent (terminal term)                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ ZFA term                                  ┃   IC ┃ genes ┃ convergent genes ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ blood vessel endothelial cell ZFA:0009036 │ 8.15 │     3 │ cdh5, fli1, kdrl │
└───────────────────────────────────────────┴──────┴───────┴──────────────────┘

  4 · grounding evidence (marker -> grounded anatomy)   
┏━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ symbol ┃ ZFA id      ┃ anatomy                       ┃
┡━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ kdrl   │ ZFA:0009036 │ blood vessel endothelial cell │
│ fli1   │ ZFA:0009036 │ blood vessel endothelial cell │
│ cdh5   │ ZFA:0009036 │ blood vessel endothelial cell │
└────────┴─────────────┴───────────────────────────────┘

               confidence rubric -> high (0.950)                
┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ component ┃ weight ┃ value ┃ bar              ┃ contribution ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ coherence │   0.40 │  1.00 │ ████████████████ │         0.40 │
│ margin    │   0.30 │  1.00 │ ████████████████ │         0.30 │
│ grounding │   0.20 │  0.75 │ ████████████     │         0.15 │
│ stage     │   0.10 │  1.00 │ ████████████████ │         0.10 │
├───────────┼────────┼───────┼──────────────────┼──────────────┤
│ score     │        │       │                  │         0.95 │
└───────────┴────────┴───────┴──────────────────┴──────────────┘

╭────────────────────────────────── Label — blood vessel endothelial cell ───────────────────────────────────╮
│ bucket           blood vessel endothelial cell   (ZFA:0009036)                                             │
│ depth            16   (evidence-derived)                                                                   │
│ panel prior      endothelium  /  mesoderm                                                                  │
│ confidence       high  (score 0.950)                                                                       │
│ components        coherence=1.00   margin=1.00   grounding=0.75   stage=1.00                               │
│ convergent genes cdh5, fli1, kdrl                                                                          │
│ next step        subcluster                                                                                │
│ rationale        blood vessel endothelial cell (IC 8.15) -- 3/4 markers converge; panel prior: endothelium │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

cardiovascular system
└── multi-tissue structure
    └── unilaminar epithelium
        └── portion of tissue
            └── vasculature
                └── simple squamous epithelium
                    └── barrier cell
                        └── epithelium
                            └── cell
                                └── vascular endothelium
                                    └── lining cell
                                        └── epithelial cell
                                            └── blood vessel endothelium
                                                └── endothelial cell
                                                    └── squamous epithelial cell
                                                        └── blood vessel endothelial cell

## Phase 3 synthesis — the handoff artifact

The deterministic annotation loop now runs end to end, and you have an instrument
(`label_and_explain`) to drive it on anything. The **handoff artifact is the `Label` evidence
packet**: a bucket (or honest abstention) at the deepest ZFA tier the evidence supports, a confidence
tier with its four-component breakdown, the convergent genes and grounded in-vivo evidence, the
coarse panel prior kept visible, and a one-line rationale — everything a human or a future LLM
narrator needs to trust or question the call, with no library internals required. (`Label.to_yaml()`
serializes the whole packet losslessly for any downstream consumer.)

That packet is the contract every later phase builds on: Phase 4b scores it against a labeled atlas,
and Phase 7's optional narrator turns it into prose.

## What’s next

**Phase 4b — evaluation.** The separate **Phase 4a** added the support-weighted anchor-rooted ZFA convergence namer —
which this notebook's `Labeler` already uses — so panels are now the coarse prior, not the naming
authority. Phase 4b runs `label()` across a labeled atlas (Daniocell’s broad tissues) and reports the
numbers that prove it works: broad-bucket agreement, resolution depth, coverage (non-abstain rate),
and abstention quality. Phase 4b measures these provisional confidence weights against ground
truth — calibrating them is deferred.

Need a refresher on the scored bucket table that feeds this notebook? Revisit
[Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 3 (with the Phase 4a namer)</b>
    <br><br>
    The full layer-1 loop: marker genes in, a grounded <code>Label</code> evidence packet out —
    labeled at the deepest ZFA anatomy term the evidence supports, with the coarse panel prior
    kept visible, rolled up or honestly abstained when evidence does not converge.
</div>

---

### Optional — design rationale: why grounding is *functions*, not a class

You may notice `ground.py` exposes plain functions (`expression_lookup`, `grounds_under`,
`stage_plausibility`) over the `expression_map` dict, rather than wrapping the expression data in a
class with methods. That is deliberate, and it is the same principle as `decide()` being pure: the
**"model" is data** (`panels.yaml` + the loaded ZFIN/ZFA maps), and the logic is a thin, testable
layer of pure functions over that data. A function that takes `(expression_map, symbol)` is trivial
to unit-test on a tiny fixture, has no hidden state, and reads top-to-bottom. A class would buy us
nothing here except indirection — the `Labeler` facade already provides the one ergonomic entry point
(`label()`) that bundles the loaded data, so the internals stay flat and boring on purpose.

*End of notebook.*